In [ ]:
import sys
sys.path.append('..')

from src.data_processor import TennisDataProcessor
from src.feature_engineer import TennisFeatureEngineer
from src.ml_dataset import TennisMLPipeline

In [ ]:
processor = TennisDataProcessor()
matches = processor.load_matches_data(data_path='../data/raw/')
clean_matches = processor.clean_matches_data()
print("Matches loaded and processed")

# Initialize feature engineer
feature_engineer = TennisFeatureEngineer(processor)
training_df = feature_engineer.create_training_dataset()
print("Feature engineer initialized")

# Initialize ML pipeline
pipeline = TennisMLPipeline(training_df)

In [ ]:
print("=== Tennis Logistic Regression Validation ===")

results = pipeline.train_and_validate()

# Display feature importance
print("Feature Importance (Logistic Regression Coefficients):")
for feature, coef in enumerate(results['feature_importance']):
    print(f"{feature:15}: {coef:6.3f}")

# Calibration check
X_test, y_test = pipeline.prepare_training_data()  # Use last 20% as test
test_size = len(X_test) // 5
X_test, y_test = X_test[-test_size:], y_test[-test_size:]

calibration = pipeline.calibration_analysis(X_test, y_test)
print(f"\nCalibration slope: {calibration['calibration_slope']:.3f} (1.0 = perfect)")
print(f"Brier score: {calibration['brier_score']:.3f} (lower = better)")

# Quick prediction example
print(f"\nExample prediction:")
test_match = processor.matches_df.iloc[10305]
test_match_features = feature_engineer.create_match_features(
    test_match['tourney_date'],
    test_match['winner_name'],
    test_match['loser_name'],
    test_match['surface']
)
print(f"{test_match['winner_name']} vs {test_match['loser_name']} probability: {pipeline.predict_match_probability(test_match_features):.2%}")
